# GK-2A 공식 관측소 좌표 기반 전처리 가이드

Rules/Data 개정본에 맞춘 학습 데이터 생성 가이드입니다. 기존 ‘외부 메타데이터 없는 16×16 타일’ 노트북은 사용하지 말고, 이 노트북의 **대회 공식 `station_list.csv` 좌표 투영** 방식을 사용하세요.

핵심 흐름은 다음과 같습니다.

1. `LE1B/KO` 16채널을 14:00 KST = 05:00 UTC로 수집
2. 공식 `LAT/LON`을 LE1B KO 격자의 row/col로 투영
3. 0km(중심), 5km, 15km 패치 특징 추출
4. 공식 좌표·고도와 날짜 특징을 결합
5. 과거 ASOS TA/HM을 target으로만 병합
6. 날짜 그룹 검증 후 직접 TA/HM 예측 모델 학습


## 1. 규정 허용/금지 표

| 항목 | 판정 | 이 노트북의 처리 |
|---|---:|---|
| GK-2A LE1B 16채널, KO | 허용 | 다운로드·특징 추출 |
| 공식 `STN_ID/LAT/LON/ALT` | 허용 | 투영·정적 특징 |
| 날짜·시각·좌표 파생값 | 허용 | 연·월·일·연중일 인코딩 |
| 과거 ASOS TA/HM | 라벨만 | `TA`, `HM` target으로만 사용 |
| ASOS lag·최근값·지점별 평균·평년값 | 금지 | 생성·저장·추론 입력 모두 제외 |
| 수치예보·ERA5·타 기관 기상자료 | 금지 | 수집하지 않음 |
| DEM·토지피복·OSM·해안선 거리 등 | 금지 | merge하지 않음 |
| `/LE1B/` 이외 API 경로 | 실격 | 정적 검사로 차단 |


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src/gk2a_weather").exists():
    raise RuntimeError("프로젝트 루트에서 노트북을 실행하세요.")
sys.path.insert(0, str(ROOT / "src"))

STATION_PATH = ROOT / "data/metadata/station_list.csv"
LABEL_PATH = ROOT / "outputs/asos_2019_2025_jja_1400kst_package/asos_2019_2025_jja_1400kst.csv"
print("project_root=", ROOT)


## 2. 대회 공식 관측소 목록 검증

열 이름·결측·중복·범위를 먼저 검사합니다. 추가 관측소명, 외부 지형, 해안선 거리 등의 열은 모델 테이블에 옮기지 않습니다.


In [ ]:
from gk2a_weather.data.stations import load_station_list

stations = load_station_list(STATION_PATH)
assert list(stations.columns) == ["STN_ID", "latitude", "longitude", "altitude"]
assert len(stations) == 96
assert stations["STN_ID"].nunique() == 96
assert not stations.isna().any().any()
print(stations.head().to_string(index=False))
print("ranges:", stations[["latitude", "longitude", "altitude"]].agg(["min", "max"]).to_dict())


## 3. 공식 좌표를 LE1B KO 픽셀로 투영

KO 영상은 Lambert Conformal Conic 격자입니다. 채널별 원본 해상도와 크기는 다음과 같습니다.

| 해상도 | 크기 | 채널 |
|---:|---:|---|
| 0.5km | 3600×3600 | VI006 |
| 1km | 1800×1800 | VI004, VI005, VI008 |
| 2km | 900×900 | 나머지 12채널 |

투영 상수는 LE1B KO NetCDF 속성과 원본 크기에서 검증한 값입니다. 만들어진 row/col은 모델 입력이 아니라, 공식 좌표에서 위성값을 찾기 위한 인덱스입니다.


In [ ]:
from gk2a_weather.constants import GK2A_CHANNELS
from gk2a_weather.features.satellite import (
    CHANNEL_RESOLUTION_KM,
    GRID_SPECS,
    official_station_pixels,
)

pixel_tables = {}
for channel in GK2A_CHANNELS:
    resolution = CHANNEL_RESOLUTION_KM[channel]
    shape = GRID_SPECS[resolution]["shape"]
    mapped = official_station_pixels(stations, channel, shape)
    assert len(mapped) == 96
    pixel_tables[channel] = mapped

print("channels=", len(pixel_tables), "stations_in_all_grids=", len(stations))
print(pixel_tables["IR105"].head().round(3).to_string(index=False))


## 4. km 단위 패치

같은 5km 반경이라도 채널 해상도에 따라 픽셀 반경이 다릅니다. 그래서 `patch_radii_km: [0, 5, 15]`를 저장하고 채널별로 자동 변환합니다.


In [ ]:
import math

radius_rows = []
for resolution in (0.5, 1.0, 2.0):
    radius_rows.append({
        "resolution_km": resolution,
        "r5km_pixels": math.ceil(5 / resolution),
        "r15km_pixels": math.ceil(15 / resolution),
    })
radius_table = pd.DataFrame(radius_rows)
print(radius_table.to_string(index=False))


## 5. 패치 특징 추출 로직 검증

아래는 2km IR105 격자와 같은 크기의 합성 배열로 관측소 96개의 중심·5km·15km 특징 추출이 끝까지 실행되는지 확인합니다. 실제 학습에서는 원본 NetCDF의 fill/scale 속성을 적용한 배열을 사용하세요.


In [ ]:
from gk2a_weather.features.satellite import extract_channel_features

rows, cols = np.indices((900, 900))
synthetic_ir105 = 280.0 + rows * 0.001 + cols * 0.002
ir105_features = extract_channel_features(
    synthetic_ir105,
    stations,
    channel="IR105",
    radii_km=(0, 5, 15),
)
assert len(ir105_features) == 96
assert ir105_features.filter(like="IR105_").notna().all().all()
print("shape=", ir105_features.shape)
print(ir105_features.head(2).round(4).to_string(index=False))


## 6. NetCDF 품질검사

각 파일에서 다음을 확인합니다.

- 요청 날짜·시각·채널과 파일 메타데이터가 일치하는가
- 2차원 영상 shape가 채널의 0.5/1/2km KO 규격과 일치하는가
- fill value, scale factor, offset이 파일 내부 속성에 따라 적용됐는가
- 유효 픽셀 비율과 물리량 범위가 비정상적이지 않은가
- 실패한 HTML/JSON 응답이나 잘린 파일을 정상 캐시로 보관하지 않았는가

파일 외부의 보정표, 수치예보, 재분석, 정적 지리정보는 사용하지 않습니다.


In [ ]:
from gk2a_weather.data.gk2a import GK2A_URL

assert "/GK2A/LE1B/" in GK2A_URL
assert "/{channel}/{area}/data" in GK2A_URL
print("LE1B path audit: PASS")


## 7. 학습 라벨 검증

ASOS 파일은 `date`, `timestamp_kst`, `STN_ID`, `TA`, `HM`만 라벨 용도로 읽습니다. `TA=-99`, `HM=-9`는 결측으로 처리하고, 다른 날짜·지점의 라벨로 임의 대체하지 않습니다.

**금지:** 일전·전일 lag, rolling mean, 지점별 계절평균, 평년값, 최근 ASOS 저장값, 평가기간 ASOS 조회.


In [ ]:
LABEL_COLUMNS = ["date", "timestamp_kst", "STN_ID", "TA", "HM"]
if LABEL_PATH.exists():
    labels = pd.read_csv(LABEL_PATH)
    assert list(labels.columns) == LABEL_COLUMNS
    assert not labels.duplicated(["date", "STN_ID"]).any()
    assert set(labels["STN_ID"].unique()) == set(stations["STN_ID"])
    assert labels["TA"].dropna().between(-50, 50).all()
    assert labels["HM"].dropna().between(0, 100).all()
    print("label_rows=", len(labels), "dates=", labels["date"].nunique())
else:
    print("학습 라벨 파일이 없어 스키립합니다:", LABEL_PATH)


## 8. 최종 학습 테이블

키는 `date × STN_ID`입니다. 입력 특징은 공식 좌표·고도, 날짜 파생값, LE1B 특징으로만 구성하고, `TA/HM`은 모델이 맞힐 target으로 분리합니다.


In [ ]:
from gk2a_weather.features.static import add_calendar_features
from gk2a_weather.features.satellite import add_channel_differences

example_day = "2019-06-01"
example = stations.merge(ir105_features, on="STN_ID", validate="one_to_one")
example.insert(0, "date", example_day)
example = add_calendar_features(example)
example = add_channel_differences(example)
if LABEL_PATH.exists():
    one_day_labels = labels.loc[labels["date"] == example_day, ["date", "STN_ID", "TA", "HM"]]
    example = example.merge(one_day_labels, on=["date", "STN_ID"], how="left", validate="one_to_one")
assert len(example) == 96
print(example.columns.tolist())
print(example.head(2).round(4).to_string(index=False))


## 9. 금지 특징 정적 검사

열 이름 검사는 보조 안전장치입니다. 최종적으로는 특징 생성 코드의 출처까지 리뷰해야 합니다.


In [ ]:
PROHIBITED_FEATURE_MARKERS = (
    "ta_lag", "hm_lag", "recent_ta", "recent_hm",
    "station_ta_mean", "station_hm_mean", "climatology", "climate_normal",
    "coast_distance", "distance_to_coast", "landcover", "terrain",
    "population", "nightlight", "soil_", "dem_", "osm_",
)

def audit_feature_columns(columns):
    violations = [
        column for column in columns
        if any(marker in column.lower() for marker in PROHIBITED_FEATURE_MARKERS)
    ]
    if violations:
        raise ValueError(f"금지 특징: {violations}")
    return True

feature_columns = [c for c in example.columns if c not in {"date", "timestamp_kst", "TA", "HM"}]
assert audit_feature_columns(feature_columns)
print("특징 열 정적 검사: PASS, count=", len(feature_columns))


## 10. 날짜 그룹 검증

한 날짜의 96개 행은 같은 위성영상을 공유하므로 행 랜덤 분할을 사용하지 않습니다. `GroupKFold(groups=date)` 또는 연도 홀드아웃으로 학습일과 검증일을 완전히 분리합니다. 평가 기간은 학습·검증·튜닝에 사용하지 않습니다.


In [ ]:
from sklearn.model_selection import GroupKFold

demo_dates = pd.date_range("2024-06-01", periods=6).repeat(2)
demo = pd.DataFrame({"date": demo_dates.astype(str), "x": np.arange(12)})
splitter = GroupKFold(n_splits=3)
for fold, (train_idx, valid_idx) in enumerate(splitter.split(demo, groups=demo["date"]), 1):
    train_dates = set(demo.iloc[train_idx]["date"])
    valid_dates = set(demo.iloc[valid_idx]["date"])
    assert train_dates.isdisjoint(valid_dates)
print("날짜 그룹 검증 교차 없음: PASS")


## 11. 학습·추론 분리

학습 노트북은 과거 ASOS 라벨을 이용해 모델을 `.fit()`하고 가중치를 저장합니다. 추론 노트북은 저장된 모델과 공식 `station_list.csv`, 평가일 LE1B/KO만 사용합니다.

추론 Dataset에서 다음을 제외하세요.

- ASOS 수집 코드·API 주소·원문·라벨 CSV
- 학습 코드·교차검증 코드
- 인증키, 로그, 외부 지리자료
- ASOS 평균·lag·평년값을 담은 어떤 파일도 제외


## 12. 최종 체크리스트

- [ ] `station_list.csv`가 공식 96행·4열이며 결측·중복이 없다.
- [ ] 16채널 모두 `LE1B`, 영역 `KO`, 14:00 KST에 맞다.
- [ ] 채널 원본 shape가 3600/1800/900 격자와 일치한다.
- [ ] 96개 지점이 모든 채널의 KO 격자 안에 있다.
- [ ] 패치 반경을 픽셀 개수가 아닌 km로 정의했다.
- [ ] ASOS TA/HM이 target 외 특징으로 사용되지 않는다.
- [ ] lag·최근값·지점별 평균·평년값·외부 지리정보 열이 없다.
- [ ] 날짜 그룹 검증을 사용했다.
- [ ] 추론 Dataset에 ASOS 수집기·라벨·학습 코드가 없다.
- [ ] 제출 전 정적 audit와 비공개 `Run All`을 둘 다 통과했다.
